In [7]:
import sys
sys.path.append(".") # make sure we can import from the current directory
from circuits import get_all_counts, get_graph_data

# run all circuits and get their counts
all_counts = get_all_counts()

# pick the "ring" circuit's counts for visualization
counts = all_counts["ring"]

# take the raw graph data
nodes_list, edges, layout = get_graph_data(counts)

# process the data into convenient formats for plotting
positions = {bs: (float(xy[0]), float(xy[1])) for bs, xy in layout.items()}
nodes = {n['id']: n['count'] for n in nodes_list}

In [ ]:
import plotly.graph_objects as go

def draw_constellation_interactive(positions, nodes, edges, title="Quantum Constellation"):
    
    total = sum(nodes.values())
    fig = go.Figure()

    # ── First layer：lines ────────────────────────────────
    for (a, b) in edges:
        if a not in positions or b not in positions:
            continue
        x0, y0 = positions[a]
        x1, y1 = positions[b]
        fig.add_trace(go.Scatter(
            x=[x0, x1, None],   # None is needed to break the line between segments
            y=[y0, y1, None],
            mode="lines",
            line=dict(color="rgba(255,255,255,0.12)", width=0.8),
            hoverinfo="none",   # no hover for lines
            showlegend=False,
        ))

    # ── Second layer：halo ────────────────────────
    halo_x, halo_y, halo_sizes, halo_colors = [], [], [], []
    for bitstring, count in nodes.items():
        if bitstring not in positions:
            continue
        prob = count / total
        n_ones = bitstring.count("1")
        halo_x.append(positions[bitstring][0])
        halo_y.append(positions[bitstring][1])
        halo_sizes.append(prob * 120 + 15)
        halo_colors.append(n_ones)

    fig.add_trace(go.Scatter(
        x=halo_x, y=halo_y,
        mode="markers",
        marker=dict(
            size=halo_sizes,
            color=halo_colors,
            colorscale="Plasma",
            opacity=0.08,
        ),
        hoverinfo="none",
        showlegend=False,
    ))

    # ── Third layer：stars (with hover) ────────────────
    star_x, star_y, star_sizes, star_colors = [], [], [], []
    hover_texts = []

    for bitstring, count in nodes.items():
        if bitstring not in positions:
            continue
        prob = count / total
        n_ones = bitstring.count("1")
        star_x.append(positions[bitstring][0])
        star_y.append(positions[bitstring][1])
        star_sizes.append(prob * 80 + 6)
        star_colors.append(n_ones)

        # hover text with rich formatting
        hover_texts.append(
            f"<b>{bitstring}</b><br>"
            f"Count: {count}<br>"
            f"Probability: {prob:.1%}<br>"
            f"Number of 1s: {n_ones}"
        )

    fig.add_trace(go.Scatter(
        x=star_x, y=star_y,
        mode="markers",
        marker=dict(
            size=star_sizes,
            color=star_colors,
            colorscale="Plasma",
            showscale=True,         # show colorbar
            colorbar=dict(
                title="No. of 1s",
                title_font=dict(color="white"),
                tickfont=dict(color="white"),
            ),
            opacity=0.9,
            line=dict(width=0),     
        ),
        text=hover_texts,
        hovertemplate="%{text}<extra></extra>",  
        showlegend=False,
    ))

    # ── layout：black background，no axes ────────────────
    fig.update_layout(
        title=dict(text=title, font=dict(color="white", size=16)),
        paper_bgcolor="black",
        plot_bgcolor="black",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False, scaleanchor="x"),  
        margin=dict(l=20, r=20, t=50, b=20),
        hoverlabel=dict(
            bgcolor="rgba(30,30,40,0.95)",
            font_color="white",
            bordercolor="rgba(255,255,255,0.2)",
        ),
        width=800,
        height=800,
    )

    return fig

In [9]:
fig = draw_constellation_interactive(
    positions,
    nodes,
    edges,
    title="Ring Entangler — real quantum data"
)
fig.show()